# 06 SVHN CNN：卷積神經網路影像分類

本 notebook 接在 `05_svhn_dnn_optimization.ipynb` 後面。

前面已經看到：Simple Dense DNN 可以明顯優於 raw-pixel 傳統 ML；Advanced Dense DNN 透過 LeakyReLU、batch size 64 與 learning rate schedule 可接近 70%。不過這些 Dense DNN 都會先把圖片攤平成一維向量，仍然沒有真正利用圖片的空間結構。

本階段改用 CNN，保留 `32x32x3` 影像形狀，透過卷積學習局部特徵。

目前本機 `ai_env` 實測結果，同一份 SVHN train 12,000 / test 3,000 子集：

| 方法 | Test Accuracy |
| --- | ---: |
| RBF SVM raw pixels + StandardScaler | 0.5170 |
| Simple Dense DNN | 0.6540 |
| Advanced Dense DNN：LeakyReLU + ReduceLROnPlateau + batch size 64 | 0.6973 |
| Small CNN from scratch | 0.9263 |

本節的核心概念是：CNN 並不只是「更深」，而是加入了更適合影像資料的 inductive bias，例如局部感受野、權重共享與空間結構保留。


## 0. 匯入套件與設定

Colab 通常已內建 TensorFlow、SciPy、scikit-learn。若缺少套件，再手動執行：

```text
%pip install scipy scikit-learn tensorflow
```


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

TRAIN_SAMPLES = 12000
TEST_SAMPLES = 3000
EPOCHS = 30
BATCH_SIZE = 128

print('TensorFlow:', tf.__version__)


## 1. 下載 SVHN cropped

SVHN 是 Street View House Numbers，本節使用 cropped 版本：每張圖片是 `32x32x3` RGB，類別是 0 到 9。

注意：Stanford 原始 `.mat` 檔的 label 用 `10` 代表數字 `0`，後面會轉成 `0`。


In [ ]:
SVHN_BASE_URL = 'http://ufldl.stanford.edu/housenumbers/'

train_path = tf.keras.utils.get_file(
    'svhn_train_32x32.mat',
    origin=SVHN_BASE_URL + 'train_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

test_path = tf.keras.utils.get_file(
    'svhn_test_32x32.mat',
    origin=SVHN_BASE_URL + 'test_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

print('train_path:', train_path)
print('test_path:', test_path)


## 2. 讀取資料並抽固定子集

為了和前面的 SVM、DNN 公平比較，此處維持相同設定：train 12,000、test 3,000，且用 stratified subset 讓每個數字類別數量接近。


In [ ]:
def load_svhn_mat(path):
    data = loadmat(path)
    X = np.transpose(data['X'], (3, 0, 1, 2))
    y = data['y'].reshape(-1) % 10
    return X, y.astype('int64')


def stratified_subset(X, y, n_samples, random_state):
    rng = np.random.default_rng(random_state)
    classes = np.unique(y)
    base = n_samples // len(classes)
    remainder = n_samples % len(classes)

    selected = []
    for rank, cls in enumerate(classes):
        cls_idx = np.where(y == cls)[0]
        take = min(base + (1 if rank < remainder else 0), len(cls_idx))
        selected.append(rng.choice(cls_idx, size=take, replace=False))

    selected = np.concatenate(selected)
    rng.shuffle(selected)
    return X[selected], y[selected]


X_train_full, y_train_full = load_svhn_mat(train_path)
X_test_full, y_test_full = load_svhn_mat(test_path)

X_train, y_train = stratified_subset(X_train_full, y_train_full, TRAIN_SAMPLES, RANDOM_STATE)
X_test, y_test = stratified_subset(X_test_full, y_test_full, TEST_SAMPLES, RANDOM_STATE + 1)

print('X_train:', X_train.shape, X_train.dtype)
print('X_test:', X_test.shape, X_test.dtype)
print('class counts:', dict(zip(*np.unique(y_train, return_counts=True))))


## 3. 觀察圖片與正規化

SVHN 是一般 8-bit RGB 影像，原始像素值是 `0-255`。神經網路訓練前先轉成 `float32` 並除以 `255.0`，縮放到 `0-1`。


In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for ax, image, label in zip(axes.ravel(), X_train[:30], y_train[:30]):
    ax.imshow(image)
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('SVHN examples')
plt.tight_layout()
plt.show()

print('pixel min:', X_train.min())
print('pixel max:', X_train.max())

X_train_float = X_train.astype('float32') / 255.0
X_test_float = X_test.astype('float32') / 255.0

print('normalized range:', X_train_float.min(), X_train_float.max())


## 4. 為什麼 CNN 比 Dense DNN 適合影像？

Dense DNN 會把 `32x32x3` 攤平成 3072 維向量，因此模型無法直接利用像素的上下左右關係。CNN 則保留圖片形狀，透過小型卷積核掃描局部區域，更容易學到筆畫、邊緣、局部紋理與數字形狀。

本範例使用小型 CNN from scratch，不使用 transfer learning，以便直接觀察 `Conv2D`、`BatchNormalization`、`MaxPooling2D` 與 `Dropout` 的角色。


In [ ]:
def build_small_svhn_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),

        tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


cnn = build_small_svhn_cnn()
cnn.summary()


## 5. 訓練 CNN

本機 CPU 實測約需 2 至 3 分鐘；在 Colab GPU 上通常會更快。若執行時間有限，可先降低 `EPOCHS`，或先參考已完成的執行結果。


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=8,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        mode='max',
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )
]

start = time.perf_counter()
history = cnn.fit(
    X_train_float,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
training_time = time.perf_counter() - start
print(f'training_time_sec: {training_time:.2f}')


## 6. 評估模型與觀察曲線

本節先以表格整理 CNN 的訓練結果，再觀察 accuracy 與 loss 曲線，用以判讀模型是否過擬合。


In [ ]:
train_loss, train_acc = cnn.evaluate(X_train_float, y_train, verbose=0)
test_loss, test_acc = cnn.evaluate(X_test_float, y_test, verbose=0)

cnn_results_df = pd.DataFrame([
    {
        'model': 'Small CNN from scratch',
        'train_acc': train_acc,
        'test_acc': test_acc,
        'best_val_acc': max(history.history['val_accuracy']),
        'epochs_ran': len(history.history['loss']),
        'training_time_sec': training_time,
    }
])
display(cnn_results_df)

history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[['accuracy', 'val_accuracy']].plot(ax=axes[0])
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
history_df[['loss', 'val_loss']].plot(ax=axes[1])
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()


## 7. 錯誤案例與混淆矩陣

錯誤案例可補充分數之外的資訊，並顯示模型最常混淆的類別與影像條件。


In [ ]:
y_proba = cnn.predict(X_test_float, verbose=0)
y_pred = y_proba.argmax(axis=1)

print(classification_report(y_test, y_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues', values_format='d')
plt.title('SVHN CNN confusion matrix')
plt.show()

wrong_idx = np.where(y_pred != y_test)[0]
print('wrong predictions:', len(wrong_idx))

fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for ax, idx in zip(axes.ravel(), wrong_idx[:24]):
    ax.imshow(X_test[idx])
    ax.set_title(f'true={y_test[idx]} pred={y_pred[idx]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 8. 以程式產生本次 CNN 結果比較

下方 code cell 會把本次 CNN 實際執行得到的 `train_acc`、`test_acc` 與訓練時間加入比較表。由於每次執行 TensorFlow 可能有小幅差異，正式課程收斂請以後面的 markdown 排名表為主。


In [ ]:
comparison = pd.DataFrame([
    {
        'method': 'RBF SVM raw pixels + StandardScaler',
        'train_acc': 0.6947,
        'test_acc': 0.5170,
        'training_time_sec': 91.65,
    },
    {
        'method': 'Simple Dense DNN',
        'train_acc': 0.7476,
        'test_acc': 0.6540,
        'training_time_sec': 22.38,
    },
    {
        'method': 'Advanced Dense DNN (LeakyReLU + LR schedule)',
        'train_acc': 0.8501,
        'test_acc': 0.6973,
        'training_time_sec': 66.61,
    },
    {
        'method': 'Small CNN from scratch',
        'train_acc': train_acc,
        'test_acc': test_acc,
        'training_time_sec': training_time,
    },
]).sort_values('test_acc', ascending=False)

display(comparison)

plt.figure(figsize=(9, 4))
plt.barh(comparison['method'], comparison['test_acc'])
plt.xlim(0, 1)
plt.xlabel('Test accuracy')
plt.title('SVHN subset: model comparison')
plt.gca().invert_yaxis()
plt.show()


## 8.1 目前 SVHN 模型總排名

以下表格使用同一份 SVHN cropped 子集進行比較：train 12,000 筆、test 3,000 筆。分數是課程目前固定的實測參考值；若重新執行 notebook，TensorFlow 模型分數可能有小幅浮動。

| 排名 | 方法 | Train Accuracy | Test Accuracy | 觀察 |
| ---: | --- | ---: | ---: | --- |
| 1 | Small CNN from scratch | 0.9814 | 0.9263 | 保留影像空間結構，準確率明顯高於 Dense DNN。 |
| 2 | Advanced Dense DNN：LeakyReLU + ReduceLROnPlateau + batch size 64 | 0.8501 | 0.6973 | 進階訓練策略可改善 Dense DNN，但仍受 flatten 表示限制。 |
| 3 | Simple Dense DNN | 0.7476 | 0.6540 | 第一個深度學習基準，已明顯優於 RBF SVM。 |
| 4 | RBF SVM raw pixels + StandardScaler | 0.6947 | 0.5170 | 傳統 ML 在真實影像 raw pixels 上測試表現受限。 |

排名顯示：同樣是 0-9 分類，只要資料從乾淨手寫數字轉成真實街景影像，模型是否保留影像空間結構會造成巨大差異。CNN 的提升不是單純因為參數更多，而是因為卷積層更符合影像資料的特性。


## 9. 課程總結

可作如下歸納：

> 同樣是 0-9 數字分類，資料從乾淨的 `load_digits` 轉為真實世界的 SVHN 後，`raw pixels` 傳統 ML 會遇到明顯瓶頸。Simple Dense DNN 與 Advanced Dense DNN 能透過非線性模型與訓練策略提升準確率，但圖片一旦攤平成向量，空間結構資訊仍會流失；CNN 則透過卷積與池化更有效地學習局部影像特徵，因此成為影像辨識的基本主流模型。

這份 notebook 的重點不是追求 SVHN 最高分，而是讓學生清楚看到模型設計如何跟資料型態相互對應。


## 課後練習與思考題

請完成下列四個任務，整理 CNN 相對於 Dense DNN 的差異。這些練習以觀察與比較為主，不一定每題都需要重新完整訓練模型。

### 練習 1：比較不同 EPOCHS 的學習趨勢

比較 `EPOCHS=10`、`20`、`30` 時可能出現的學習趨勢。這題可以先用觀察與推論回答，不一定要重新完整訓練。

#### 練習 1 參考答案：比較不同 EPOCHS 的觀察方式

| Epochs | 預期現象 | 檢查重點 |
| ---: | --- | --- |
| 10 | 訓練較快，但可能尚未收斂。 | validation accuracy 是否仍在明顯上升。 |
| 20 | 通常比 10 epochs 更穩定。 | train/test score 是否同步改善。 |
| 30 | 有機會更高，也可能開始停滯。 | 若 `val_loss` 上升但 `train_loss` 下降，要注意過擬合。 |

Epochs 不是越大越好，應搭配 validation curve、EarlyStopping 與 test score 判斷。


### 練習 2：拿掉 Dropout 後觀察模型

建立拿掉 Dropout 的 CNN 版本，先觀察模型參數量，並思考 train/test score 可能如何改變。

In [ ]:
# 練習 2 參考答案：建立拿掉 Dropout 的 CNN 版本，觀察模型參數量

def build_cnn_without_dropout():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

no_dropout_model = build_cnn_without_dropout()
pd.DataFrame([{
    'model': 'CNN without Dropout',
    'params': no_dropout_model.count_params(),
}])


#### 練習 2 參考答案：拿掉 Dropout 後的預期影響

| 模型 | Regularization | 預期影響 |
| --- | --- | --- |
| CNN with Dropout | 有 Dropout | 降低過擬合風險，訓練可能稍慢但泛化較穩定。 |
| CNN without Dropout | 無 Dropout | `train_acc` 可能較快升高，但 `test_acc` 不一定跟著提升。 |

拿掉 Dropout 後若 `train_acc` 明顯高於 `test_acc`，代表模型可能更容易記住訓練資料。


### 練習 3：比較第一層 filters 數量

比較第一層卷積 filters 數量為 16、32、64 時，模型參數量如何變化。

In [ ]:
# 練習 3 參考答案：比較第一層 filters 數量與參數量

def build_cnn_with_first_filters(first_filters=32):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(first_filters, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    return model

filter_rows = []
for first_filters in [16, 32, 64]:
    model_variant = build_cnn_with_first_filters(first_filters)
    filter_rows.append({
        'first_conv_filters': first_filters,
        'total_params': model_variant.count_params(),
        'input_shape': '(32, 32, 3)',
    })

filter_df = pd.DataFrame(filter_rows)
display(filter_df)



#### 練習 3 參考答案：比較第一層 filters 數量與參數量

- 觀察說明：filters 越多，模型可學的局部特徵種類越多，但參數量與計算量也會增加。
- 是否值得增加 filters，仍需回到 validation/test score 來判斷。


### 練習 4：比較 Dense DNN 與 CNN 的輸入表示

比較 Dense DNN 與 CNN 的輸入表示方式，說明 CNN 為什麼更適合影像資料。這題偏向觀念整理，不需要額外程式。

#### 練習 4 參考答案：比較 Dense DNN 與 CNN 的輸入表示

| 模型 | 模型看到的輸入 | 空間結構 | 優點 | 限制 |
| --- | --- | --- | --- | --- |
| Dense DNN | 3072 維向量 | 一開始就攤平成一維，鄰近像素關係不明確。 | 容易理解，適合銜接 Dense layer 概念。 | 影像局部特徵與位置關係利用不足。 |
| CNN | `32 x 32 x 3` image tensor | 保留高度、寬度與色彩通道。 | 卷積可學習局部邊緣、筆畫與形狀特徵。 | 架構概念較多，需要理解 filters 與 feature maps。 |

SVHN 是真實照片資料，CNN 的架構假設更符合影像的空間特性，因此通常比 Dense DNN 更適合這類任務。
